In [33]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [34]:
df = pd.read_csv(
    r"C:/Sachin/project/Ola Bike Rides Request Demand Forecast/data/raw_data.csv", 
    low_memory=False,
    compression="gzip"
)

In [35]:
len(df)

8381556

In [36]:
df.head()

,ts,number,pick_lat,pick_lng,drop_lat,drop_lng
0,2020-03-26 07:07:17,14626,12.313621,76.658195,12.287301,76.602280
1,2020-03-26 07:32:27,85490,12.943947,77.560745,12.954014,77.543770
2,2020-03-26 07:36:44,05408,12.899603,77.587300,12.934780,77.569950
3,2020-03-26 07:38:00,58940,12.918229,77.607544,12.968971,77.636375
4,2020-03-26 07:39:29,05408,12.899490,77.587270,12.934780,77.569950


In [37]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8381556 entries, 0 to 8381555
Data columns (total 6 columns):
 #   Column    Dtype  
---  ------    -----  
 0   ts        str    
 1   number    str    
 2   pick_lat  float64
 3   pick_lng  float64
 4   drop_lat  float64
 5   drop_lng  float64
dtypes: float64(4), str(2)
memory usage: 383.7 MB


In [38]:
df.shape

(8381556, 6)

### A Customer_ID `number` at a particular timestamp can only have one entry
### Removing Duplicate Entries ['ts','number']

In [39]:
df[df.duplicated(subset=['ts','number'], keep=False)]

,ts,number,pick_lat,pick_lng,drop_lat,drop_lng
235,2020-03-26 18:10:35,16795,12.967236,77.641594,13.014504,77.650856
236,2020-03-26 18:10:35,16795,12.967236,77.641594,13.014504,77.650856
407,2020-03-26 21:35:50,65856,12.917173,77.586400,12.913940,77.685280
408,2020-03-26 21:35:50,65856,12.917173,77.586400,12.913940,77.685280
443,2020-03-26 23:26:29,27554,12.933715,77.619300,12.938208,77.587520
...,...,...,...,...,...,...
8381231,2021-03-26 22:23:12,61636,12.975229,77.620370,13.017285,77.618200
8381245,2021-03-26 22:25:13,61636,12.975229,77.620370,13.017285,77.618200
8381246,2021-03-26 22:25:13,61636,12.975229,77.620370,13.017285,77.618200
8381248,2021-03-26 22:25:27,61636,12.975229,77.620370,13.017285,77.618200


### There are 113540 Duplicate Entries
#### We have 8315498 Unique timestamp, customer_id rows. 

In [40]:
## Keeping first occurence

df.drop_duplicates(subset=['ts','number'], keep ='first', inplace = True)

df.reset_index(drop = True, inplace = True)

In [41]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8315498 entries, 0 to 8315497
Data columns (total 6 columns):
 #   Column    Dtype  
---  ------    -----  
 0   ts        str    
 1   number    str    
 2   pick_lat  float64
 3   pick_lng  float64
 4   drop_lat  float64
 5   drop_lng  float64
dtypes: float64(4), str(2)
memory usage: 380.7 MB


In [42]:
#Count missing values
np.count_nonzero(df.isnull().values)

np.int64(0)

In [43]:
df['number'] = pd.to_numeric(df['number'], errors='coerce')

#Count missing values
np.count_nonzero(df.isnull().values)

np.int64(116)

#### There are 116 NaN rows, dropping NaN rows.

In [44]:
df.dropna(inplace = True)
len(df)

8315382

In [45]:
df['number'] = pd.to_numeric(df['number'], errors='coerce', downcast='integer')
df['ts'] = pd.to_datetime(df['ts'], errors='coerce')

In [46]:
df.info()

<class 'pandas.DataFrame'>
Index: 8315382 entries, 0 to 8315497
Data columns (total 6 columns):
 #   Column    Dtype         
---  ------    -----         
 0   ts        datetime64[us]
 1   number    int32         
 2   pick_lat  float64       
 3   pick_lng  float64       
 4   drop_lat  float64       
 5   drop_lng  float64       
dtypes: datetime64[us](1), float64(4), int32(1)
memory usage: 412.4 MB


### Breaking Time to Features

In [47]:
df['mins'] = df['ts'].dt.minute
df['hour'] = df['ts'].dt.hour
df['day'] = df['ts'].dt.day 
df['month'] = df['ts'].dt.month
df['year'] = df['ts'].dt.year   
df['dayofweek'] = df['ts'].dt.dayofweek

In [48]:
df

,ts,number,pick_lat,pick_lng,drop_lat,drop_lng,mins,hour,day,month,year,dayofweek
0,2020-03-26 07:07:17,14626,12.313621,76.658195,12.287301,76.602280,7,7,26,3,2020,3
1,2020-03-26 07:32:27,85490,12.943947,77.560745,12.954014,77.543770,32,7,26,3,2020,3
2,2020-03-26 07:36:44,5408,12.899603,77.587300,12.934780,77.569950,36,7,26,3,2020,3
3,2020-03-26 07:38:00,58940,12.918229,77.607544,12.968971,77.636375,38,7,26,3,2020,3
4,2020-03-26 07:39:29,5408,12.899490,77.587270,12.934780,77.569950,39,7,26,3,2020,3
...,...,...,...,...,...,...,...,...,...,...,...,...
8315493,2021-03-26 23:55:24,50410,12.907856,77.557870,12.954270,77.530785,55,23,26,3,2021,4
8315494,2021-03-26 23:58:15,12580,12.981010,77.694450,12.969070,77.704280,58,23,26,3,2021,4
8315495,2021-03-26 22:11:20,72339,12.924252,77.650520,12.905820,77.630570,11,22,26,3,2021,4
8315496,2021-03-26 22:12:30,72339,12.924252,77.650520,12.905820,77.630570,12,22,26,3,2021,4


In [49]:
"df.to_csv('C:/Sachin/project/preprocessed_1.csv',index = False, compression = 'gzip')"

"df.to_csv('C:/Sachin/project/preprocessed_1.csv',index = False, compression = 'gzip')"

In [50]:
from geopy.distance import geodesic
from geopy.geocoders import Nominatim
import time
geolocator = Nominatim(user_agent="OLABikes")

## 🔹 Data Cleaning with Business Understanding

In real-world ride booking systems, raw data often contains duplicate requests, user errors, and invalid entries.  
Instead of applying only basic preprocessing, we clean the data using business logic to ensure the dataset reflects **true ride demand**.

---

### 🔸 Case 1: Rebooking to Same Location (Within 1 Hour)

Users may rebook rides due to long waiting time, driver cancellations, or app/network issues.  
This results in multiple requests for the same trip, inflating demand.

**Approach:**
- For the same user and pickup location, retain only **one request within a 1-hour window**

**Impact:**
- Removes duplicate ride intent  
- Prevents overestimation of demand  

---

### 🔸 Case 2: Incorrect Location Entry (Within 8 Minutes)

Users may accidentally enter incorrect pickup/drop locations and quickly rebook with corrected details.

**Approach:**
- If multiple bookings are made within **8 minutes**, retain only the **latest request**

**Assumption:**
- 8 minutes approximates a short bike ride duration, indicating quick correction

**Impact:**
- Eliminates incorrect entries  
- Preserves the final and correct user intent  

---

### 🔸 Case 2.1: Unrealistic Short Trips (< 50 meters)

Trips where pickup and drop locations are extremely close are likely due to data errors, GPS noise, or accidental/test bookings.

**Approach:**
- Remove trips with distance **< 0.05 km (50 meters)**

**Impact:**
- Removes non-meaningful trips  
- Improves overall data quality  

---

### 🔸 Case 3: Outside Service Area

Some ride requests may fall outside the operational region of the service.

**Approach:**
- Filter data using **latitude-longitude bounding box** to keep only valid service areas

**Impact:**
- Ensures the model learns from realistic and serviceable locations  

---

## ✅ Summary

This step ensures:
- Duplicate and misleading requests are removed  
- User mistakes are corrected  
- Only valid and meaningful ride data is retained  

---

⚠️ Note: The thresholds used (1 hour, 8 minutes, 50 meters) are based on domain assumptions and can be further validated using data analysis or business inputs.

#### Check lat-long bounding box coordinates

In [51]:
df['ts'] = pd.to_datetime(df['ts'])

In [52]:
df.sort_values(by = ['number','ts'], inplace = True)

In [53]:
df.reset_index(drop=True, inplace=True)

In [54]:
df

,ts,number,pick_lat,pick_lng,drop_lat,drop_lng,mins,hour,day,month,year,dayofweek
0,2020-10-10 07:34:16,-1,12.975773,77.571070,12.878468,77.445330,34,7,10,10,2020,5
1,2020-10-11 08:23:42,-1,12.930813,77.609530,12.960320,77.587210,23,8,11,10,2020,6
2,2020-10-11 08:23:50,-1,12.930813,77.609530,12.960320,77.587210,23,8,11,10,2020,6
3,2020-10-11 08:23:51,-1,12.930813,77.609530,12.960320,77.587210,23,8,11,10,2020,6
4,2020-10-11 08:23:54,-1,12.930813,77.609530,12.960320,77.587210,23,8,11,10,2020,6
...,...,...,...,...,...,...,...,...,...,...,...,...
8315377,2021-02-12 19:37:11,99999,13.029848,77.593400,13.063751,77.589850,37,19,12,2,2021,4
8315378,2021-02-19 20:43:25,99999,13.029296,77.592580,12.927923,77.627106,43,20,19,2,2021,4
8315379,2021-02-20 17:34:45,99999,12.907576,77.600685,12.925874,77.607620,34,17,20,2,2021,5
8315380,2021-02-27 08:26:23,99999,12.956665,77.521870,12.948099,77.562990,26,8,27,2,2021,5


In [55]:
# you need convert first to numpy array by values and cast to int64 - output is in nanosecond, so need divide by 10 ** 9

df['booking_timestamp'] = df['ts'].astype('int64') // 10**9

## 🔹 Why This is Useful

### ✅ 1. ML Models Prefer Numerical Values
Machine learning models require numerical inputs. Converting datetime into numeric form makes it suitable for model training.

---

### ✅ 2. Easy Time Difference Calculation
Numeric or datetime formats allow efficient computation of time differences, which is essential for analyzing user behavior and identifying patterns like rebookings.

---

### ✅ 3. Faster Computation
Numerical operations are computationally more efficient than datetime operations, improving performance when working with large datasets.

In [56]:
df.head(20)

,ts,number,pick_lat,pick_lng,drop_lat,drop_lng,mins,hour,day,month,year,dayofweek,booking_timestamp
0,2020-10-10 07:34:16,-1,12.975773,77.571070,12.878468,77.445330,34,7,10,10,2020,5,1602315
1,2020-10-11 08:23:42,-1,12.930813,77.609530,12.960320,77.587210,23,8,11,10,2020,6,1602404
2,2020-10-11 08:23:50,-1,12.930813,77.609530,12.960320,77.587210,23,8,11,10,2020,6,1602404
3,2020-10-11 08:23:51,-1,12.930813,77.609530,12.960320,77.587210,23,8,11,10,2020,6,1602404
4,2020-10-11 08:23:54,-1,12.930813,77.609530,12.960320,77.587210,23,8,11,10,2020,6,1602404
5,2020-10-11 08:23:56,-1,12.930813,77.609530,12.960320,77.587210,23,8,11,10,2020,6,1602404
6,2020-10-11 11:57:17,-1,12.960213,77.587460,12.930824,77.609610,57,11,11,10,2020,6,1602417
7,2020-10-11 11:57:31,-1,12.960213,77.587460,12.930824,77.609610,57,11,11,10,2020,6,1602417
8,2020-10-16 17:51:07,-1,12.924353,77.549410,12.932216,77.581825,51,17,16,10,2020,4,1602870
9,2020-10-16 17:51:25,-1,12.924353,77.549410,12.932216,77.581825,51,17,16,10,2020,4,1602870


In [57]:
df['time_diff_sec'] = df.groupby('number')['booking_timestamp'].diff()

df['booking_time_diff_hr'] = df['time_diff_sec'] // 3600
df['booking_time_diff_min'] = df['time_diff_sec'] // 60

df[['booking_time_diff_hr', 'booking_time_diff_min']] = \
df[['booking_time_diff_hr', 'booking_time_diff_min']].fillna(0)

## 🔹 Why This Feature is Important

### 1. Captures User Activity
The time gap between bookings reflects user activity levels.  
Short gaps indicate frequent usage, while longer gaps suggest inactivity.

---

### 2. Helps in Demand Prediction
This feature helps the model learn:
- When a user is likely to book again  
- Patterns of peak usage and demand cycles  

---

### 3. Real-World Relevance
This is commonly known as a **Recency Feature**.

It is widely used in:
- Ride-hailing platforms (e.g., Uber, Ola)  
- E-commerce platforms (last purchase behavior)  
- Recommendation systems  

In [58]:
df

,ts,number,pick_lat,pick_lng,drop_lat,drop_lng,mins,hour,day,month,year,dayofweek,booking_timestamp,time_diff_sec,booking_time_diff_hr,booking_time_diff_min
0,2020-10-10 07:34:16,-1,12.975773,77.571070,12.878468,77.445330,34,7,10,10,2020,5,1602315,NaN,0.0,0.0
1,2020-10-11 08:23:42,-1,12.930813,77.609530,12.960320,77.587210,23,8,11,10,2020,6,1602404,89.0,0.0,1.0
2,2020-10-11 08:23:50,-1,12.930813,77.609530,12.960320,77.587210,23,8,11,10,2020,6,1602404,0.0,0.0,0.0
3,2020-10-11 08:23:51,-1,12.930813,77.609530,12.960320,77.587210,23,8,11,10,2020,6,1602404,0.0,0.0,0.0
4,2020-10-11 08:23:54,-1,12.930813,77.609530,12.960320,77.587210,23,8,11,10,2020,6,1602404,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8315377,2021-02-12 19:37:11,99999,13.029848,77.593400,13.063751,77.589850,37,19,12,2,2021,4,1613158,90.0,0.0,1.0
8315378,2021-02-19 20:43:25,99999,13.029296,77.592580,12.927923,77.627106,43,20,19,2,2021,4,1613767,609.0,0.0,10.0
8315379,2021-02-20 17:34:45,99999,12.907576,77.600685,12.925874,77.607620,34,17,20,2,2021,5,1613842,75.0,0.0,1.0
8315380,2021-02-27 08:26:23,99999,12.956665,77.521870,12.948099,77.562990,26,8,27,2,2021,5,1614414,572.0,0.0,9.0


In [59]:
##Booking time different in mins
df['booking_time_diff_min'].value_counts().to_dict()

{0.0: 6233136,
 1.0: 681469,
 2.0: 290695,
 4.0: 148657,
 3.0: 130265,
 5.0: 108979,
 7.0: 66255,
 8.0: 59919,
 6.0: 52265,
 10.0: 41527,
 9.0: 34799,
 11.0: 33833,
 12.0: 24190,
 14.0: 22861,
 15.0: 20103,
 13.0: 18722,
 17.0: 17362,
 18.0: 16798,
 20.0: 14970,
 21.0: 12639,
 16.0: 12575,
 19.0: 10571,
 24.0: 10017,
 23.0: 8740,
 22.0: 8395,
 25.0: 8352,
 27.0: 8298,
 30.0: 7707,
 28.0: 7647,
 31.0: 6458,
 26.0: 6239,
 34.0: 5413,
 29.0: 5169,
 33.0: 5055,
 37.0: 4774,
 40.0: 4703,
 38.0: 4311,
 32.0: 4202,
 35.0: 4189,
 41.0: 3976,
 36.0: 3931,
 44.0: 3457,
 43.0: 3366,
 39.0: 3143,
 50.0: 3105,
 47.0: 3066,
 46.0: 2743,
 48.0: 2628,
 42.0: 2588,
 51.0: 2557,
 45.0: 2416,
 54.0: 2399,
 53.0: 2340,
 60.0: 2213,
 49.0: 2213,
 57.0: 2190,
 56.0: 1993,
 63.0: 1753,
 59.0: 1743,
 61.0: 1740,
 70.0: 1690,
 52.0: 1671,
 64.0: 1668,
 67.0: 1650,
 58.0: 1609,
 55.0: 1596,
 66.0: 1525,
 69.0: 1376,
 62.0: 1329,
 73.0: 1294,
 80.0: 1264,
 74.0: 1246,
 76.0: 1224,
 65.0: 1212,
 71.0: 1152,
 77.0

In [60]:
##Booking time different in hours
df['booking_time_diff_hr'].value_counts().to_dict()

{0.0: 8216989,
 1.0: 55991,
 2.0: 19758,
 3.0: 10619,
 4.0: 5906,
 5.0: 3314,
 6.0: 1867,
 7.0: 808,
 8.0: 130}

## 🔹 Why This is Important

### 1. Understand User Behavior
Analyzing the distribution of time differences helps identify user activity patterns, such as frequent bookings, regular usage intervals, and inactive users.

---

### 2. Demand Insights
The distribution provides insights into how often bookings occur over time, helping to understand short-term demand intensity and usage frequency patterns.

---

### 3. Feature Validation
This step ensures the feature is logically consistent and reliable by checking for invalid values (e.g., negative time gaps) and identifying potential noise or anomalies.

In [61]:
# valid_df = df[df['time_diff_sec'] > 0]

# less_than_1hr = (valid_df['booking_time_diff_hr'] == 0).sum()
# greater_equal_1hr = (valid_df['booking_time_diff_hr'] >= 1).sum()

# total = len(valid_df)
# print(f"Total valid bookings: {total}")
# print(f"Percentage of bookings with time difference less than 1 hour: {less_than_1hr / total * 100:.2f}%")
# print(f"Percentage of bookings with time difference greater than or equal to 1 hour: {greater_equal_1hr / total * 100:.2f}%")


In [62]:
len(df)

8315382

In [63]:
### Handling Case 1: Re-booking Again to Same Location within 1hour by same user

df = df[~((df.duplicated(subset=['number','pick_lat','pick_lng'],keep=False)) & (df.booking_time_diff_hr<=1))]

In [64]:
## Before removing Row Count
len(df)

3227549

### We observe that there are 8315382 - 3227549 = 50,87,833 booking that happen in less than 1 hour of request by a user

###  Removed 50,87,833 rows in `Case1` we now have 32,27,549

In [65]:
df.to_csv('C:/Sachin/project/preprocessed_2.csv',index = False, compression = 'gzip')